In [ ]:
import os

# Working directory must contain AlphaSimPy.py for imports
os.chdir(r"/Users/mtwatson/Library/CloudStorage/Box-Box/Projects/AI agent for breeding/Endpoint 2 agent")


# da_Silva_Plant_Breeding_Program - AlphaSimPy Translation

This notebook converts the provided **BRAID breeding program abstraction** into a tutorial-style **AlphaSimPy** simulation.

The source BRAID program describes a simple plant line-breeding workflow:
- two founders (`P1` and `P2`)
- a biparental crossing block of about 300 families
- advancement through **F1**, **F2**, and **single-seed-descent-like selfing**
- evaluation of **F2:4-derived material** with phenotyping and genotyping
- truncation selection of the top lines

## BRAID-informed assumptions

Several values in the BRAID abstraction were placeholders or not fully specified in the source diagram. To make the notebook runnable, the following assumptions are used:

1. A small but nonzero genome is simulated with `runMacs`.
2. One additive trait is simulated with broad-sense intent approximated through phenotyping error control.
3. The two named founders are represented by sampling two individuals from a simulated founder population.
4. The crossing block is implemented as repeated biparental crossing between the two founders.
5. The `50 SSD per population` note is interpreted as **50 F2-derived lines per family** after SSD advancement.
6. The F2:4 evaluation stage uses phenotypes plus a simple genomic-information placeholder based on true genetic values, because the BRAID file specifies GS/EBV conceptually but not a concrete prediction model.
7. Selection intensity is implemented as the top **10%** of evaluated lines.

This notebook is structured in the style of the AlphaSimPy tutorials.

## Import Required Libraries

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from AlphaSimPy import (
    runMacs,
    SimParam,
    newPop,
    randCross,
    self,
    setPheno,
    selectInd,
    meanG,
    varG,
)

print('AlphaSimPy BRAID Translation Notebook')
print('Libraries imported successfully.')


## Program Parameters

These parameters are derived from the BRAID abstraction where possible, with explicit defaults for missing values.

In [ ]:
# ---- BRAID-derived program settings ----
program_name = 'da_Silva_Plant_Breeding_Program'
timestep_unit = 'year'
horizon = 4

# ---- Genome assumptions for a runnable AlphaSimPy example ----
ploidy = 2
n_chr = 1
seg_sites = 1000
n_qtl = 100
n_snp = 200
heritability = 0.3

# ---- Population sizes from BRAID ----
n_founders = 2
n_crosses = 300
parents_per_cross = 2
n_f1 = 300
n_f2 = 300
ssd_lines_per_family = 50   # assumption from BRAID note: '50 SSD per population'
n_selected = int(n_crosses * ssd_lines_per_family * 0.10)

# ---- Phenotyping / evaluation settings ----
n_env = 1
n_rep = 1
error_variance = 1.0

print('Program:', program_name)
print('Horizon:', horizon, timestep_unit + 's')
print('Genome:')
print('  ploidy =', ploidy)
print('  chromosomes =', n_chr)
print('  QTL per chromosome =', n_qtl)
print('  SNP per chromosome =', n_snp)
print('  heritability =', heritability)
print('Workflow sizes:')
print('  founders =', n_founders)
print('  crossing block / F1 families =', n_crosses)
print('  SSD lines per family =', ssd_lines_per_family)
print('  selected lines =', n_selected)


## Create Founder Haplotypes and Simulation Parameters

The BRAID file does not specify a genetic map or chromosome count, so a compact one-chromosome example is used here.

In [ ]:
# Simulate founder haplotypes
founderPop = runMacs(nInd=20, nChr=n_chr, segSites=seg_sites, inbred=True)

# Create simulation parameters
SP = SimParam(founderPop)
SP.addTraitA(nQtlPerChr=n_qtl)
SP.setVarE(h2=heritability)
SP.addSnpChip(nSnpPerChr=n_snp)

print('Founder haplotypes simulated.')
print('SimParam object created with one additive trait and one SNP chip.')


## Define the Two BRAID Founders

The BRAID abstraction names two external founders, `P1` and `P2`. Here they are represented by selecting two individuals from the simulated founder population.

In [ ]:
founders = newPop(founderPop, simParam=SP)
parents = selectInd(founders, nInd=n_founders, use='gv', simParam=SP)

print('Two founders selected to represent P1 and P2.')
print('Founder genetic mean:', meanG(parents))
print('Founder genetic variance:', varG(parents))


## Stage 1: Biparental Crossing Block

BRAID node `n1_cross` specifies a biparental crossing block with approximately 300 crosses from the two founders.

Because only two founders are available, the crossing block is implemented as repeated crossing between them.

In [ ]:
crossing_block = randCross(parents, nCrosses=n_crosses, nProgeny=1, simParam=SP)

print('Crossing block created.')
print('Number of F1 individuals:', crossing_block.nInd)
print('Crossing block meanG:', meanG(crossing_block))
print('Crossing block varG:', varG(crossing_block))


## Stage 2: Advance to F1 and F2 by Selfing

The BRAID workflow advances the crossing block through one generation of selfing to F1 and another generation of selfing to F2.

In [ ]:
# In this simplified implementation, the crossing block individuals are treated as the F1 generation
f1 = crossing_block

# Self each F1 once to create one F2 individual per family
f2 = self(f1, nProgeny=1, simParam=SP)

print('F1 and F2 stages created.')
print('F1 nInd:', f1.nInd)
print('F2 nInd:', f2.nInd)
print('F2 meanG:', meanG(f2))
print('F2 varG:', varG(f2))


## Stage 3: SSD Advancement to F2:4-derived Material

BRAID node `n4_advance_ssd` specifies two generations of selfing from F2 to produce F2-derived SSD material. Here this is implemented as:

- self F2 to F3 with multiple progeny per family
- self again to F4

Using 50 lines per family yields a total evaluation set of `300 x 50 = 15,000` lines.

In [ ]:
# Self F2 to produce multiple F3 lines per family
f3 = self(f2, nProgeny=ssd_lines_per_family, simParam=SP)

# Self F3 once more to obtain F2:4-derived material
f2_4_test = self(f3, nProgeny=1, simParam=SP)

print('SSD advancement complete.')
print('F3 nInd:', f3.nInd)
print('F2:4 test population nInd:', f2_4_test.nInd)
print('F2:4 meanG:', meanG(f2_4_test))
print('F2:4 varG:', varG(f2_4_test))


## Stage 4: Phenotyping and Genomic-Information Placeholder

BRAID node `n5_evaluate_f2_4` specifies phenotyping and genotyping of F2:4 material.

This notebook uses:
- `setPheno` to generate phenotypes
- a simple selection index combining phenotype and true genetic value as a placeholder for the unspecified genomic prediction model

This keeps the notebook runnable while preserving the BRAID intent of combining phenotypic and genomic information.

In [ ]:
setPheno(f2_4_test, varE=error_variance, simParam=SP)

# Build a simple BRAID-inspired index using phenotype + genomic information placeholder
# Here, true genetic value is used as a stand-in for EBV because the BRAID file does not define a prediction model.
pheno_values = np.array(f2_4_test.pheno).reshape(-1)
gv_values = np.array(f2_4_test.gv).reshape(-1)
selection_index = 0.5 * pheno_values + 0.5 * gv_values

summary_df = pd.DataFrame({
    'pheno': pheno_values,
    'gv': gv_values,
    'index': selection_index,
})

print(summary_df.describe())


## Stage 5: Select the Top 10% of Lines

BRAID node `n6_select_top_lines` specifies truncation selection with intensity `0.1` at the individual level.

Because AlphaSimPy selection functions typically use built-in criteria, this notebook ranks lines externally using the BRAID-inspired index and then reports the selected subset summary.

In [ ]:
n_keep = max(1, int(len(summary_df) * 0.10))
selected_df = summary_df.sort_values('index', ascending=False).head(n_keep).copy()

print('Number evaluated:', len(summary_df))
print('Number selected:', len(selected_df))
print('\nSelected lines summary:')
print(selected_df.describe())


## Results Summary

The BRAID outputs requested tracking of:
- genetic mean
- genetic variance
- inbreeding

This notebook directly reports genetic mean and variance by stage. Inbreeding is not explicitly calculated here because the exact AlphaSimPy helper for that metric may vary by installation, but the repeated selfing structure clearly increases homozygosity across stages.

In [ ]:
results = pd.DataFrame([
    {'stage': 'founders', 'nInd': parents.nInd, 'meanG': float(meanG(parents)), 'varG': float(varG(parents))},
    {'stage': 'crossing_block', 'nInd': crossing_block.nInd, 'meanG': float(meanG(crossing_block)), 'varG': float(varG(crossing_block))},
    {'stage': 'f1', 'nInd': f1.nInd, 'meanG': float(meanG(f1)), 'varG': float(varG(f1))},
    {'stage': 'f2', 'nInd': f2.nInd, 'meanG': float(meanG(f2)), 'varG': float(varG(f2))},
    {'stage': 'f2_4_test', 'nInd': f2_4_test.nInd, 'meanG': float(meanG(f2_4_test)), 'varG': float(varG(f2_4_test))},
])

print(results)


## Visualization

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].plot(results['stage'], results['meanG'], marker='o')
axes[0].set_title('Genetic Mean by Stage')
axes[0].set_xlabel('Stage')
axes[0].set_ylabel('meanG')
axes[0].tick_params(axis='x', rotation=45)

axes[1].plot(results['stage'], results['varG'], marker='o', color='darkorange')
axes[1].set_title('Genetic Variance by Stage')
axes[1].set_xlabel('Stage')
axes[1].set_ylabel('varG')
axes[1].tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.show()


## Conclusion

This notebook provides a direct AlphaSimPy translation of the BRAID abstraction for the **da Silva** breeding program.

It preserves the main workflow logic:
- biparental crossing
- selfing to F2
- SSD-style advancement to F2:4-derived lines
- phenotyping and genomic-information-assisted ranking
- top-line selection

If a more detailed genomic prediction model, explicit family structure, or exact source-diagram counts become available, this notebook can be extended accordingly.